In [93]:
run_id = 1780201885
env = 'dev'

In [94]:
from databricks.connect.session import DatabricksSession as SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import *
from datetime import datetime, timedelta
import yaml

with open('/home/ken/projects/box-office/src/box_office/config.yaml', 'r') as file:
    config = yaml.safe_load(file).get(env)
catalog = config.get('catalog')
franchises_config = config.get('showtimes')
showtimes_snapshot_folderpath = config.get('showtimes').get('showtimes_snapshot_folderpath').format(run_id=run_id)

In [95]:
spark = SparkSession.builder.profile("main").serverless().getOrCreate()

In [96]:
df = spark.read.json(showtimes_snapshot_folderpath)

# Movies

In [ ]:
# Update movies DB with info about new movies

movies_new = ( df
    .select(F.explode('viewModel.movies'))
    .select(
          'col.id'
        , 'col.genres'
        , 'col.darkPoster'
        , 'col.mopURI'
        , 'col.poster'
        , 'col.rating'
        , 'col.releaseDate'
        , 'col.runtime'
        , 'col.title'
        )
    .withColumn('rn', F.row_number().over(Window.partitionBy('id').orderBy(F.lit(1))))
    .filter('rn = 1')
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [ ]:
target_fqn = f'{catalog}.base.movies'
movies_new.limit(0).write.mode('ignore').saveAsTable(target_fqn)

(
    movies_new
    .mergeInto(target_fqn, movies_new.id == F.col('movies.id'))
        .whenMatched().updateAll()
        .whenNotMatched().insertAll()
        .merge()
)

# Showtimes

In [ ]:
# Update showtimes with info about new showtimes
import pyspark.sql.functions as F

showtimes_new  = (df
    .selectExpr(
        'viewModel.theater.details.id as theater_id'
        , 'explode(viewModel.movies) as movies'
        )
    .selectExpr(
        '* except (movies)'
        , 'movies.id as movie_id'
        , 'explode(movies.variants) as variants'
    )
    .selectExpr(
        '* except (variants)'
        , 'variants.amenityGroupClassName'
        , 'variants.filmFormatHeader'
        , 'variants.filmFormatHeaderClassName'
        , 'explode(variants.amenityGroups) as amenityGroups'
        )
    .selectExpr(
         '* except(amenityGroups)'
        , 'transform(amenityGroups.amenities, x -> x.id) as amenity_ids'
        , 'cast(amenityGroups.hasReservedSeating as boolean) as hasReservedSeating'
        , 'cast(amenityGroups.isDolby as boolean) as isDolby'
        , 'amenityGroups.lateNightMsg'
        , 'amenityGroups.movieVariantId'
        , 'explode(amenityGroups.showtimes) as showtimes'
        )
    .selectExpr(
         'showtimes.id'
        , 'showtimes.showtimeHashCode'
        , 'to_utc_timestamp(to_timestamp(showtimes.ticketingDate, "yyyy-MM-dd+HH:mm"), "America/Los_Angeles") as ticketingDate'
        , 'showtimes.date as time'
        , 'cast(showtimes.hasMatineeMessage as boolean) as hasMatineeMessage'
        , 'showtimes.message'
        , 'showtimes.screenReaderTime'
        , 'showtimes.ticketingJumpPageURL'
        , '* except (showtimes)'
        )
    .filter(F.col('id').isNotNull())
    .withColumn('rn', F.row_number().over(Window.partitionBy('id').orderBy(F.lit(1))))
    .filter('rn = 1')
    .drop('rn')
)

1348296


In [ ]:
# Quick validations
__showtimes_new = showtimes_new.toPandas()

num_showtimes = __showtimes_new.shape[0]
assert (num_showtimes > 1250000) and (num_showtimes <= 1500000)

assert min(__showtimes_new['ticketingDate']) > datetime.now() - timedelta(days=3)

num_showings_per_day = __showtimes_new.assign(ticketingDate=__showtimes_new['ticketingDate'].dt.date).groupby('ticketingDate').size().reset_index(name='count')['count'].mean()
assert num_showings_per_day > 40000 and num_showings_per_day <= 50000 

num_theaters = __showtimes_new['theater_id'].nunique()
assert (num_theaters > 2500) and (num_theaters<3000)

In [ ]:
target_fqn = f'{catalog}.base.showtimes'
showtimes_new.limit(0).write.mode('ignore').saveAsTable(target_fqn)

(
    showtimes_new
    .mergeInto(target_fqn, showtimes_new.id == F.col('showtimes.id'))
        .whenMatched().updateAll()
        .whenNotMatched().insertAll()
        .merge()
)

# Amenities

In [ ]:
# Update amenities with new info

amenities_new  = (df
    .select(F.explode('viewModel.movies').alias('movies'))
    .select(F.explode('movies.variants').alias('variants'))
    .select(F.explode('variants.amenityGroups').alias('amenityGroups'))
    .select(F.explode('amenityGroups.amenities').alias('amenities'))
    .selectExpr(
        'amenities.id'
        , 'amenities.description'
        , 'amenities.imageUrl'
        , 'amenities.name'
        )
    .filter(F.col('id').isNotNull())
    .withColumn('rn', F.row_number().over(Window.partitionBy('id').orderBy(F.lit(1))))
    .filter('rn = 1')
    .drop('rn')   
)
amenities_new.show()

+----+--------------------+--------------------+--------------------+
|  id|         description|            imageUrl|                name|
+----+--------------------+--------------------+--------------------+
|1002|IMAX® digital pro...|{{https://images....|                IMAX|
|1003|Open captioning s...|                NULL|        Open caption|
|1004|Spanish subtitles...|                NULL|   Spanish subtitled|
|1005|IMAX® 3D presents...|                NULL|             IMAX 3D|
|1006|This is the VIP a...|                NULL|                 VIP|
|1007|Standard format m...|                NULL|        GIANT SCREEN|
|1009|RealD is the most...|{{https://images....|            RealD 3D|
|1012|Closed Captioning...|                NULL|      Closed caption|
|1014|Digital 3D descri...|                NULL|          Digital 3D|
|1017|Cinemark XD audit...|                NULL|         Cinemark XD|
|1019|RPX auditoriums f...|                NULL|                 RPX|
|1020|Sensory Friend

In [ ]:
target_fqn = f'{catalog}.base.amenities'
amenities_new.limit(0).write.saveAsTable(target_fqn, mode='ignore')

(
    amenities_new.alias('amenities_new')
    .mergeInto(target_fqn, amenities_new.id == F.col('amenities.id'))
        .whenMatched().updateAll()
        .whenNotMatched().insertAll()
        .merge()
)